In [ ]:
# Standard libraries
import json
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from functools import partial
from itertools import product

# Specific libraries
import nibabel as nib
import seaborn as sns

# Import custom modules
import file_io as fio
import dicom_load as dcmload
import registration as reg
import temperature as temp
import denoise as dn

In [ ]:
# Define directories
CALIB_DIR = Path("./scanner_2")
SCAN_DIR = CALIB_DIR / "scan_data"
TEMP_DIR = CALIB_DIR / "temperature_data"
OUTPUT_DIR = CALIB_DIR / "derivatives"
testname = CALIB_DIR.stem

if not CALIB_DIR.is_dir():
    raise NotADirectoryError(f"Expected a directory, got: {CALIB_DIR}") 
if not SCAN_DIR.is_dir():
    raise NotADirectoryError(f"{CALIB_DIR.stem} doesn't contain subdirectory named {SCAN_DIR.stem}") 
if not TEMP_DIR.is_dir():
    raise NotADirectoryError(f"{CALIB_DIR.stem} doesn't contain subdirectory named {TEMP_DIR.stem}") 

OUTPUT_DIR.mkdir(exist_ok=True)

### LOAD & UNPACK

In [ ]:
# Load in data and save basic features 
dicom_fnames = fio.scrape_files(SCAN_DIR, filetype=["MRe", "dcm"])
dicom_fnames.sort(key = lambda x: dcmload.dicom_scantime(x))
print(f"The full dataset contains {len(dicom_fnames)} DICOM files")
dataset_unpacked = list(map(dcmload.process_dicom, dicom_fnames))
dataset_df = pd.DataFrame(dataset_unpacked)
print("Scans unpacked into pandas DataFrame structure")

### PAIR W/ TEMP DATA

In [ ]:
# Now pair scans with their temperature measurements and remove any scans without valid temperature measurements
temp_fname = TEMP_DIR / "temperatures.csv"
temp_log = np.loadtxt(temp_fname, dtype=str, delimiter=",", skiprows=1)
temp_log_dt = [[dcmload.convert_timestamp(datetime_string), float(temp)] for datetime_string, temp in temp_log]
temp_log_dt_np = np.array(temp_log_dt)

dataset_df["temperature"] = dataset_df["scan_time"].apply(partial(temp.estimate_temp, temp_log=temp_log_dt, time_window_minutes=90, f_past=0.5, n_min=None))
print(f"Full dataset contains {dataset_df.shape[0]} scans")

n_lacking_temp = np.count_nonzero(dataset_df["temperature"].isna())
print(f"Removing {n_lacking_temp} scans - don't have corresponding temperature data")

valid_df = dataset_df[dataset_df["temperature"].notna()].copy()
print(f"'Temperature trimmed' dataset contains {valid_df.shape[0]} scans")

fig, ax = plt.subplots(1, 1, figsize=(20,8))
ax.scatter(temp_log_dt_np[:,0], 
           temp_log_dt_np[:,1], 
           marker="x", 
           linewidth=1 , 
           c="k", 
           label="Raw temperature measurements")

ax.scatter(valid_df["scan_time"].values, 
           valid_df["temperature"].values, 
           marker=".", 
           linewidth=0.1, 
           c="r",
           label="Estimated phantom measurement during scan")
ax.legend()
ax.set_ylabel("Temperature (C)")
ax.set_xlabel("Time")
fig.tight_layout()
plt.show()

In [ ]:
def plot_num_of_repeats(df, col_name):

    num_repeats = df.groupby(col_name).size().rename("count").reset_index()

    fig, ax = plt.subplots(figsize=(16, 9))
    sns.barplot(data=num_repeats, x=col_name, y="count", hue=col_name, palette="husl", ax=ax)

    # Tidy up axes
    ax.set_xticks(np.arange(num_repeats.values.shape[0]), num_repeats[col_name])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center")
    ax.set_yticks(range(0, num_repeats["count"].values.max() + 1))
    ax.set_ylabel("Number of Repeats")
    fig.tight_layout()

    plt.show()

plot_num_of_repeats(valid_df, "seq_name")


### MPPCA DENOISING
MP-PCA is used only to make the registration reliable. The denoised scans are not used for the calibration.

In [ ]:
"""
# ----------------------------------
# RUN ON SUBGROUPS INDEPENDENTLY
# ----------------------------------

df_list = []
n_iter = len(valid_df["dwi_grad_duration"].unique())

for i, dur in enumerate(valid_df["dwi_grad_duration"].unique(), start=1):

    print(f"Denoising iteration: {i} \/ {n_iter}")
    # Extract scans from dataframe
    graddur_df = valid_df[valid_df["dwi_grad_duration"] == dur].copy()
    raw_scans = np.stack(graddur_df["image"].values, axis=-1)

    denoised_scans = dn.apply_mppca(raw_scans, default_kernel=True, kernel=None)
    graddur_df["mppca_image"] = list(np.transpose(denoised_scans, [3,0,1,2]))
    df_list.append(graddur_df)

denoised_df = pd.concat(df_list).reset_index(drop=True)
denoised_df.sort_values(by="scan_time", axis="index", ignore_index=True, inplace=True)
"""

# ----------------------------------
# RUN ON FULL DATASET SIMULTANEOUSLY
# ----------------------------------

raw_scans = np.stack(valid_df["image"].values, axis=-1)
denoised_scans = dn.apply_mppca(raw_scans, default_kernel=False, kernel="7,7,7")
denoised_df = valid_df.copy()
denoised_df["mppca_image"] = list(np.transpose(denoised_scans, [3,0,1,2]))

### REGISTRATION
Transformations are estimated on the MP-PCA denoised scans and then applied to the original noisy scans.

In [ ]:
# Extract images from dataframe
raw_4d = np.stack(denoised_df["image"].values, axis=-1)
mppca_4d = np.stack(denoised_df["mppca_image"].values, axis=-1)
aff_0 = denoised_df["affine"].values[0]
spacing_0 = reg.spacing_from_affine(aff_0)

# Extract b-values in order to be able to identify b=0 scan to register other scans to
bvals = denoised_df["b_value"].values

# Perform registration:
# The registration transformations are estimated on the MP-PCA denoised scans, as the high-b noisy scans
# don't have enough SNR to be registered reliably.
# The function returns a registered 4D image array, a numpy array of indices for which registration failed
# and the list of transformations (one per scan).
reg_mppca_4d, i_drop, transforms = reg.register_to_b0(data_4d=mppca_4d,
                                                      bvals=bvals,
                                                      spacing=spacing_0,
                                                      b0_threshold=5,
                                                      return_transforms=True)

# The same transformations are then applied to the original (noisy) scans.
reg_raw_4d = reg.apply_registration(data_4d=raw_4d,
                                    transforms=transforms,
                                    spacing=spacing_0)

# Create a new dataframe with only the measurements that were successfully
# registered and append columns for the registered scans and the transformations used
reg_df = denoised_df.drop(index=i_drop).reset_index(drop=True)
reg_df["reg_mppca_image"] = list(np.transpose(reg_mppca_4d, (3,0,1,2))) # registered, denoised
reg_df["reg_image"] = list(np.transpose(reg_raw_4d, (3,0,1,2)))         # registered, noisy
reg_df["transform"] = [t for t in transforms if t is not None]

unreg_df = denoised_df.loc[i_drop].reset_index(drop=True)

### CREATE b=0 MASK

In [ ]:
b0_threshold = 5

# Mask is created from the registered denoised b=0 scans

b0_scans = np.stack(reg_df[reg_df["b_value"] < b0_threshold]["reg_mppca_image"].values, axis=-1)
b0_avg = np.mean(b0_scans, axis=-1)
b0_mask = reg.get_mask(b0_avg, lt_percentile=55)

#px.imshow(b0_mask, animation_frame=2, zmin=0, zmax=1, height=800, width=1000, title="b=0 mask")

In [ ]:
# This cell is for plotting the standard deviation across repeats for each sequence
# pre and post registration to check we are seeing an improvement in as expected
def voxel_std_plot(df, cat_1, cat_2, groupby="seq_name", mask=None):

    rows = []

    if mask is None:
        _mask = np.ones_like(df[cat_1].values[0]).astype(bool)
    else:
        _mask = mask.astype(bool)

    for sn, subset_df in df.groupby(groupby):
        
        img_1 = np.stack(subset_df[cat_1].values)
        img_2  = np.stack(subset_df[cat_2].values)
        axis = subset_df["dwi_direction"].unique()[0]

        std_1 = np.std(img_1, axis=0)
        std_2  = np.std(img_2, axis=0)

        std_1_vals = std_1[_mask]
        std_2_vals = std_2[_mask]

        rows.append(pd.DataFrame({
            "seq_name": sn,
            "stage": str(cat_1),
            "std_value": std_1_vals,
            "dwi_direction": axis
        }))

        rows.append(pd.DataFrame({
            "seq_name": sn,
            "stage": str(cat_2),
            "std_value": std_2_vals,
            "dwi_direction": axis
        }))

    plot_df = pd.concat(rows, ignore_index=True)

    dwi_axes = plot_df["dwi_direction"].unique()
    for a in dwi_axes:

        subset_df = plot_df[plot_df["dwi_direction"]==a].copy()
        plt.figure(figsize=(20, 8))

        ax = sns.boxplot(
            data=subset_df,
            x="seq_name",
            y="std_value",
            hue="stage",
            palette="husl",
            showfliers=False
        )

        ax.tick_params(axis="x", rotation=90)
        ax.set_xlabel("Sequence")
        ax.set_ylabel("Voxelwise STD")
        ax.set_title(f"DWI Direction = {a}")

        plt.tight_layout()
        plt.show()

voxel_std_plot(reg_df, cat_1="image", cat_2="reg_image", mask=b0_mask)

## SAVE OUTPUTS

In [ ]:
# Original (noisy) and MP-PCA denoised scans, before registration
raw_images = np.stack(reg_df["image"].values, axis=-1)
raw_nii = nib.Nifti1Image(raw_images, affine=aff_0)
fio.save_nifti(raw_nii, OUTPUT_DIR, testname + "_" + "RAW.nii.gz")

dn_images = np.stack(reg_df["mppca_image"].values, axis=-1)
dn_nii = nib.Nifti1Image(dn_images, affine=aff_0)
fio.save_nifti(dn_nii, OUTPUT_DIR, testname + "_" + "RAW_MPPCA.nii.gz")

# Registered scans (transformations estimated on the MP-PCA scans and applied to both sets).
# RAW_REG (noisy) is the dataset used for the analysis.
reg_images = np.stack(reg_df["reg_image"].values, axis=-1)
reg_nii = nib.Nifti1Image(reg_images, affine=aff_0)
fio.save_nifti(reg_nii, OUTPUT_DIR, testname + "_" + "RAW_REG.nii.gz")

reg_dn_images = np.stack(reg_df["reg_mppca_image"].values, axis=-1)
reg_dn_nii = nib.Nifti1Image(reg_dn_images, affine=aff_0)
fio.save_nifti(reg_dn_nii, OUTPUT_DIR, testname + "_" + "RAW_MPPCA_REG.nii.gz")

b0_mask_nii = nib.Nifti1Image(b0_mask.astype(float), aff_0)
fio.save_nifti(b0_mask_nii, OUTPUT_DIR, testname + "_" + "MASK.nii.gz")

# Registration transformations, in the same row order as the parameters file
transforms_fname = OUTPUT_DIR / (testname + "_" + "transforms.json")
with open(transforms_fname, "w") as f:
    json.dump(reg_df["transform"].tolist(), f)

param_df = reg_df.drop(["image", "mppca_image", "reg_image", "reg_mppca_image", "transform"], axis=1, inplace=False)
param_fname = OUTPUT_DIR / (testname + "_" + "params.h5")
param_df.to_hdf(param_fname, key="df", mode="w", index=False)